In [106]:
import os
import json
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool,InjectedToolArg
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [29]:
load_dotenv()

True

In [30]:
currency_conversion_api_key = os.getenv("CURRENCY_CONVERSION_API_KEY")

In [56]:
@tool
def get_conversion_rate(base_currency:str , target_currency: str) -> dict:
    '''This fucntion will take base currency and target currecny as input and it will return the latest resl time
    conversion rate from the api. The conversion rate will be float in nature.'''
    url = f"https://v6.exchangerate-api.com/v6/{currency_conversion_api_key}/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

In [95]:
@tool
def convert(base_currency_amount: int,conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    ''' given a currency conversion rate this function calculates the target currency value from a given base currency value'''
    return base_currency_amount * conversion_rate

In [96]:
get_conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786233601,
 'time_last_update_utc': 'Sun, 09 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1786320001,
 'time_next_update_utc': 'Mon, 10 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2492}

# Executing Tool Calling Steps

In [150]:
llm = ChatOpenAI()

In [151]:
llm_with_tools = llm.bind_tools(tools = [get_conversion_rate,get_currency_value])

In [152]:
messages = [HumanMessage('What is the conversion factor between Pounds and INR, and based on that can you convert 10 pounds to inr')]

In [153]:
ai_response = llm_with_tools.invoke(messages)

In [154]:
messages.append(ai_response)

In [155]:
ai_response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 150, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAyL7OjxPe7iNa6XxembryO55vVLc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe6d1-c0af-7a41-8203-047d478434aa-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'GBP', 'target_currency': 'INR'}, 'id': 'call_FnANYHWKmJocz6dV4eDfapGy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 22, 'total_tokens': 172, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_toke

In [156]:
tool_call1 = ai_response.tool_calls[0]

In [157]:
tool_message1 = get_conversion_rate.invoke(tool_call1)

In [158]:
tool_message_converted = json.loads(tool_message1.content)

In [159]:
messages.append(tool_message1)

In [160]:
messages

[HumanMessage(content='What is the conversion factor between Pounds and INR, and based on that can you convert 10 pounds to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 150, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAyL7OjxPe7iNa6XxembryO55vVLc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe6d1-c0af-7a41-8203-047d478434aa-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'GBP', 'target_currency': 'INR'}, 'id': 'call_FnANYHWKmJocz6dV4eDfapGy', 'type': 'tool_c

In [161]:
ai_response = llm_with_tools.invoke(messages)

In [162]:
messages.append(ai_response)

In [163]:
tool_call2 = ai_response.tool_calls[0]

In [164]:
tool_call2

{'name': 'get_currency_value',
 'args': {'base_currency_amount': 10},
 'id': 'call_gmPzjnKGw0pAag7Z6v1Tvamp',
 'type': 'tool_call'}

In [165]:
tool_message_converted

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786233601,
 'time_last_update_utc': 'Sun, 09 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1786320001,
 'time_next_update_utc': 'Mon, 10 Aug 2026 00:00:01 +0000',
 'base_code': 'GBP',
 'target_code': 'INR',
 'conversion_rate': 128.2535}

In [166]:
conversion_rate = tool_message_converted['conversion_rate']

In [167]:
tool_call2['args']['conversion_rate'] = conversion_rate

In [168]:
tool_call2

{'name': 'get_currency_value',
 'args': {'base_currency_amount': 10, 'conversion_rate': 128.2535},
 'id': 'call_gmPzjnKGw0pAag7Z6v1Tvamp',
 'type': 'tool_call'}

In [169]:
tool_message2 = convert.invoke(tool_call2)

In [170]:
tool_message2

ToolMessage(content='1282.535', name='convert', tool_call_id='call_gmPzjnKGw0pAag7Z6v1Tvamp')

In [171]:
messages.append(tool_message2)

In [172]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between Pounds (GBP) and Indian Rupees (INR) is 128.2535. \n\nTherefore, 10 Pounds would be equivalent to 1282.535 Indian Rupees.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 354, 'total_tokens': 399, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAyLn4qVhtFZ5nk6UXRDjKsg8OR14', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe6d2-60c3-7df0-910e-7dba5cbaec21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 354, 'output_tokens': 45, 'total_tokens': 399, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_deta